In [1]:
# Import required libraries
import pandas as pd
import numpy as np 
import math 
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import os
import glob
import re
from PIL import Image
from pathlib import Path
import base64, io

In [5]:
# Load all per-participant *_wTurns.csv files and combine into one dataset

# source folder with the per-participant files created earlier
turns_folder = os.path.join(os.path.dirname(folder), "Data with Turns")

# collect files
files = sorted(glob.glob(os.path.join(turns_folder, "*_wTurns.csv")))
pattern = re.compile(r'(\d{4})_(\d)_wTurns\.csv$', re.IGNORECASE)

dfs = []
for fp in files:
    df_i = pd.read_csv(fp)
    m = pattern.search(os.path.basename(fp))
    if m:
        pid, sess = map(int, m.groups())
        if 'SubjectID' not in df_i.columns:
            df_i['SubjectID'] = pid
        if 'Session' not in df_i.columns:
            df_i['Session'] = sess
    dfs.append(df_i)

combined = pd.concat(dfs, ignore_index=True, sort=False)

out_path = os.path.join(turns_folder, "combined_dataframe.csv")
combined.to_csv(out_path, index=False)

print(f"Loaded {len(dfs)} file(s) from {turns_folder}")
print(f"Saved → {out_path}")

Loaded 29 file(s) from C:/Users/Marcel/Desktop/Study Project/VR Data\Data with Turns
Saved → C:/Users/Marcel/Desktop/Study Project/VR Data\Data with Turns\combined_dataframe.csv


In [2]:
# load
csv_path = r"C:\Users\Marcel\Desktop\Study Project\VR Data\Data with Turns\combined_dataframe.csv"
df = pd.read_csv(csv_path)

# coerce needed columns
df["events"]  = pd.to_numeric(df["events"], errors="coerce")
df["length"]  = pd.to_numeric(df["length"], errors="coerce")            # seconds
df["combinedGazeValidityBitmask"] = pd.to_numeric(df["combinedGazeValidityBitmask"], errors="coerce")

# threshold (58.4 ms in seconds)
THRESH_S = 0.0584

# mask: valid saccades only
mask = (
    (df["events"] == 1.0) &
    df["long_events"].notna() &
    (df["combinedGazeValidityBitmask"] != 0) &
    df["length"].notna()
)

# create column with NA elsewhere
df["saccade_mode"] = pd.Series(pd.NA, index=df.index, dtype="Int64")
df.loc[mask, "saccade_mode"] = np.where(df.loc[mask, "length"] < THRESH_S, 1, 2)

# optional quick check
print(df["saccade_mode"].value_counts(dropna=False))

saccade_mode
<NA>    3127248
2         76266
1         66501
Name: count, dtype: Int64


In [4]:
out_path = r"C:\Users\Marcel\Desktop\Study Project\VR Data\Data with Turns\combined_dataframe.csv"
df.to_csv(out_path, index=False)
print(f"Saved → {out_path}")

Saved → C:\Users\Marcel\Desktop\Study Project\VR Data\Data with Turns\combined_dataframe.csv
